# Sauvegarde du Modèle avec BentoML

## Objectif
Ce notebook démontre comment sauvegarder et déployer le **modèle Random Forest gagnant** du projet P6 Seattle Energy Prediction en utilisant **BentoML**.

---

## 1️. Import des Librairies Requises

In [90]:
# Import des librairies essentielles
import pandas as pd
import numpy as np
import bentoml
from datetime import datetime

# Import Scikit-learn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## 2️. Chargement et Préparation des Données

In [91]:
# Chargement des données alignées sur le notebook 05 (X/y déjà préparés)
X = pd.read_csv('../sources/2016_Building_Energy_Benchmarking_03X_building_consumption.csv')
y = pd.read_csv('../sources/2016_Building_Energy_Benchmarking_03y_building_consumption.csv')

print("\n" + "=" * 100)
print("📊 Chargement des données (X/y alignés avec code 05)...")
print("=" * 100 + "\n")
print(f"✅ X: {X.shape[0]} lignes, {X.shape[1]} colonnes")
print(f"✅ y: {y.shape}")
try:
    print(f"📝 Colonnes X (extrait): {list(X.columns)[:12]}{' …' if X.shape[1] > 12 else ''}")
except Exception:
    pass

# Normaliser y en Series si DataFrame à une colonne
if isinstance(y, pd.DataFrame):
    if y.shape[1] == 1:
        y = y.iloc[:, 0]
    else:
        # Si plusieurs colonnes, tenter de sélectionner la cible attendue
        tgt_candidates = [c for c in y.columns if 'SiteEnergyUse(kWh)' in c or 'SiteEnergyUse' in c]
        y = y[tgt_candidates[0]] if tgt_candidates else y.iloc[:, 0]

print(f"Cible y type: {type(y)}; longueur: {len(y)}")


📊 Chargement des données (X/y alignés avec code 05)...

✅ X: 1277 lignes, 36 colonnes
✅ y: (1277, 1)
📝 Colonnes X (extrait): ['PropertyGFATotal', 'NumberofFloors', 'Age_batiment', 'Categorie_age', 'Consommation_par_etage', 'Surface_par_etage', 'Densite_energetique', 'Taille_batiment', 'Hauteur_batiment', 'Surface_x_Age', 'PrimaryPropertyType_Hospital', 'PrimaryPropertyType_Hotel'] …
Cible y type: <class 'pandas.core.series.Series'>; longueur: 1277


In [92]:
# Préparation des features et target (version sans fuite)
import re

# Définir/rappeler la cible
target = 'SiteEnergyUse(kWh)'

# Garde anti-fuite: exclure les colonnes de consommation/énergie directes ou dérivées
risk_pattern = re.compile(r'(electric|kwh|kbtu|naturalgas|gas|steam|fuel|energyuse|consumption|eui|ghg)', re.IGNORECASE)

print("\n" + "=" * 100)
print("Préparation des données d'entraînement (anti-fuite) sur X issu de code 05...")
print("=" * 100 + "\n")

# Retirer toute colonne risquée de X
safe_cols = [c for c in X.columns if not risk_pattern.search(c)]

# Retirer la cible de X si présente (par sécurité)
if target in safe_cols:
    safe_cols.remove(target)

X = X[safe_cols].copy()

# Sanity check
print(f"Features utilisées (sans fuite): {len(X.columns)}")
print(f"Extrait: {list(X.columns)[:12]}{' …' if X.shape[1] > 12 else ''}")
print(f"Target: {target}")

# Nettoyage des données (drop lignes avec NaN sur X ou y)
print("\n" + "=" * 100)
print("Nettoyage des données...")
print("=" * 100 + "\n")
print(f"Valeurs manquantes X (total): {int(X.isnull().sum().sum())}")
print(f"Valeurs manquantes y: {int(pd.isnull(y).sum())}")

mask = (~X.isnull().any(axis=1)) & (~pd.isnull(y))
X = X[mask]
y = y[mask]

print(f"Données nettoyées: {len(X)} échantillons")
print(f"Forme finale X: {X.shape}")
print(f"Forme finale y: {y.shape if hasattr(y, 'shape') else len(y)}")

# Exposer la liste de features pour la sauvegarde BentoML
available_features = list(X.columns)


Préparation des données d'entraînement (anti-fuite) sur X issu de code 05...

Features utilisées (sans fuite): 36
Extrait: ['PropertyGFATotal', 'NumberofFloors', 'Age_batiment', 'Categorie_age', 'Consommation_par_etage', 'Surface_par_etage', 'Densite_energetique', 'Taille_batiment', 'Hauteur_batiment', 'Surface_x_Age', 'PrimaryPropertyType_Hospital', 'PrimaryPropertyType_Hotel'] …
Target: SiteEnergyUse(kWh)

Nettoyage des données...

Valeurs manquantes X (total): 0
Valeurs manquantes y: 0
Données nettoyées: 1277 échantillons
Forme finale X: (1277, 36)
Forme finale y: (1277,)


## 3️. Sélection du Modèle Random Forest

In [93]:
# Division des données
print("\n" + "=" * 100)
print("Division des données...")
print("=" * 100 + "\n")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train set: {X_train.shape[0]} échantillons")
print(f"Test set: {X_test.shape[0]} échantillons")

# Entraînement du modèle Random Forest avec hyperparamètres fournis (pas de recherche)
print("\n" + "=" * 100)
print("Random Forest — paramètres fournis")
print("=" * 100 + "\n")

provided_params = {
    'n_estimators': 150,
    'max_depth': 30,
    'min_samples_split': 2,
    'min_samples_leaf': 2,
    'max_features': 1.0,
    'bootstrap': True,
    'random_state': 42,
    'n_jobs': -1
}

rf_model = RandomForestRegressor(**provided_params)
rf_model.fit(X_train, y_train)

# Évaluation du modèle
y_pred_train = rf_model.predict(X_train)
y_pred_test = rf_model.predict(X_test)

r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)
mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

print(f"Performance sur Train:")
print(f"   • R² Score: {r2_train:.4f} ({r2_train*100:.2f}%)")
print(f"   • MAE: {mae_train:,.0f}")
print(f"   • RMSE: {rmse_train:,.0f}")

print(f"\nPerformance sur Test:")
print(f"   • R² Score: {r2_test:.4f} ({r2_test*100:.2f}%)")
print(f"   • MAE: {mae_test:,.0f}")
print(f"   • RMSE: {rmse_test:,.0f}")

# Assigner le modèle retenu à 'model' pour la suite (sauvegarde BentoML)
model = rf_model

# Paramètres pour les métadonnées (types simples uniquement)
safe_best_params = {k: (v if v is not None else "None") for k, v in provided_params.items()}

# Stockage des métriques pour BentoML
model_metrics = {
    "algorithm": "Random Forest Regressor",
    "best_params": safe_best_params,
    "r2_train": float(r2_train),
    "r2_test": float(r2_test),
    "mae_train": float(mae_train),
    "mae_test": float(mae_test),
    "rmse_train": float(rmse_train),
    "rmse_test": float(rmse_test),
    "n_features": len(available_features),
    "n_train_samples": len(X_train),
    "n_test_samples": len(X_test)
}


Division des données...

Train set: 1021 échantillons
Test set: 256 échantillons

Random Forest — paramètres fournis

Performance sur Train:
   • R² Score: 0.9938 (99.38%)
   • MAE: 28,397
   • RMSE: 79,901

Performance sur Test:
   • R² Score: 0.9600 (96.00%)
   • MAE: 71,482
   • RMSE: 203,966


## 4️. Sauvegarde du Modèle avec BentoML

In [94]:
# 🗑️ Nettoyage des anciens modèles
print("\n" + "=" * 100)
print("Nettoyage des anciens modèles...")
print("=" * 100 + "\n")

try:
    # Lister tous les modèles existants avec le même nom
    existing_models = [m for m in bentoml.models.list() if m.tag.name == "seattle_energy_predictor"]
    
    if existing_models:
        print(f"Trouvé {len(existing_models)} ancien(s) modèle(s) à supprimer:")
        for old_model in existing_models:
            try:
                print(f"  Suppression: {old_model.tag}")
                bentoml.models.delete(old_model.tag)
                print(f"  Supprimé: {old_model.tag}")
            except Exception as e:
                print(f"  Échec suppression {old_model.tag}: {str(e)}")
    else:
        print("Aucun ancien modèle à nettoyer")
        
except Exception as e:
    print(f"Erreur lors du nettoyage: {str(e)}")
    print("Continuons avec la sauvegarde...")

# Préparation des métadonnées
metadata = {
    "project": "P6_Seattle_Energy_Prediction",
    "algorithm": "Random Forest Regressor",
    "performance": {
        "r2_score": f"{r2_test:.4f} ({r2_test*100:.2f}%)",
        "mae_test": f"{mae_test:,.0f} kWh",
        "rmse_test": f"{rmse_test:,.0f} kWh"
    },
    "features": available_features,
    "target": target,
    "training_date": datetime.now().isoformat(),
    "model_version": "v1.0.0",
    "author": "ABAI_P6",
    "best_params": safe_best_params
}

# Labels pour identification
labels = {
    "project": "seattle_energy_prediction",
    "algorithm": "random_forest",
    "version": "1.0.0",
    "stage": "production"
}

print("\n" + "=" * 100)
print("Sauvegarde du nouveau modèle...")
print("=" * 100 + "\n")

# Sauvegarde avec BentoML
bentoml_model = bentoml.sklearn.save_model(
    name="seattle_energy_predictor",
    model=model,
    labels=labels,
    metadata=metadata,
    custom_objects={
        "feature_names": available_features,
        "model_metrics": model_metrics,
        "sample_input": X_test.iloc[0].to_dict()
    }
)

print(f"Modèle sauvegardé avec succès!")
print(f"Tag BentoML: {bentoml_model.tag}")
print(f"Path: {bentoml_model.path}")
print(f"Labels: {bentoml_model.info.labels}")


Nettoyage des anciens modèles...

Trouvé 1 ancien(s) modèle(s) à supprimer:
  Suppression: seattle_energy_predictor:2pqcqg52i6vjepak
  Supprimé: seattle_energy_predictor:2pqcqg52i6vjepak

Sauvegarde du nouveau modèle...

Modèle sauvegardé avec succès!
Tag BentoML: seattle_energy_predictor:4srpmh52i6jxcpak
Path: C:\Users\aymer\AppData\Local\Temp\bentoml-model-seattle_energy_predictor-9dmcclw4
Labels: {'project': 'seattle_energy_prediction', 'algorithm': 'random_forest', 'version': '1.0.0', 'stage': 'production'}
Modèle sauvegardé avec succès!
Tag BentoML: seattle_energy_predictor:4srpmh52i6jxcpak
Path: C:\Users\aymer\AppData\Local\Temp\bentoml-model-seattle_energy_predictor-9dmcclw4
Labels: {'project': 'seattle_energy_prediction', 'algorithm': 'random_forest', 'version': '1.0.0', 'stage': 'production'}


## 5️. Vérification du Modèle Sauvegardé

In [95]:
# Vérification du modèle dans le store BentoML
print("\n" + "=" * 100)
print("Vérification du modèle sauvegardé...")
print("=" * 100 + "\n")

# Vérification basique du modèle
print(f"Vérification du modèle: {bentoml_model.tag}")
loaded_model = bentoml.models.get(bentoml_model.tag)
print("Modèle accessible dans le store BentoML!")

print(f"Labels: {loaded_model.info.labels}")
print(f"Créé le: {loaded_model.info.creation_time}")

print("\nLe modèle est correctement sauvegardé et accessible!")


Vérification du modèle sauvegardé...

Vérification du modèle: seattle_energy_predictor:4srpmh52i6jxcpak
Modèle accessible dans le store BentoML!
Labels: {'project': 'seattle_energy_prediction', 'algorithm': 'random_forest', 'version': '1.0.0', 'stage': 'production'}
Créé le: 2025-11-05 13:03:58.981783+00:00

Le modèle est correctement sauvegardé et accessible!
